### Tuning for automatic parameter optimization
In this notebook, we will explore the tuning functionalities in Qewton. 
Since in this workshop multiple people work on the same GPUs, we use a simple toy example for this demonstration.

The problem is given by the ODE:
\begin{align*}
\partial_t u &= 4*\pi*\cos(4*\pi*t) + 2*t \quad \text{in } (0,2) \\
u(0) &= 0 
\end{align*}

In [ ]:
# This block is for GPU selection. Please execute.
# import pathlib
# import os
# user = int(str(pathlib.Path().resolve())[22:24])
# os.environ["CUDA_VISIBLE_DEVICES"]= str(user % 4)

In [ ]:
import qewton
import math 

# Variables like always
T = qewton.Variable("t", dim=1)
U = qewton.Variable("u", dim=1)

# The interval also
interval = qewton.geometries.Interval(T, 0, 2)

In [ ]:
inner_sampler = qewton.RandomUniformSampler(
        interval, n_points=qewton.DiscreteHyperparameter((100, 5000), name="Inner Sampler Points")
    )
left_sampler = qewton.GridSampler(interval.boundary_left, 1)

In [ ]:
model = qewton.FCN(
    in_neurons=T,
    hidden_neurons=qewton.DiscreteHyperparameter((4, 64)),
    out_neurons=U,
    n_hidden_layers=qewton.DiscreteHyperparameter((1, 5)),
    activation=qewton.bb.Tanh,
)

In [ ]:
cos = qewton.bb.Cos()

def residual_fun(u: U, t: T):
    return u.gradient(t) - (4.0*math.pi*cos(4.0*math.pi*t) + 2.0*t)

constraint = qewton.PINNConstraint(residual_fun, name="PINNConstraint")
ode_graph = qewton.PINNPipeline(inner_sampler, [model], constraint)

In [ ]:
def boundary_residual_fun(u: U):
    return u

boundary_constraint = qewton.constraints.PINNConstraint(
    boundary_residual_fun, 
    name="InitialConstraint", 
    weight=qewton.ContinuousHyperparameter((1, 100), name="Boundary Weight")
)
boundary_graph = qewton.PINNPipeline(left_sampler, [model], boundary_constraint)

In [ ]:
validation_constraint = qewton.MSEConstraint(name="ValidationConstraint", 
                                             evaluated_in_mode=qewton.EvaluationPhase.VALIDATION)
validation_pipeline = qewton.Graph()
Sin = qewton.bb.Sin()

def real_solution(t: T):  # type: ignore
    return (Sin(4.0*math.pi*t) + t**2).as_variable(U)

with validation_pipeline.tracker(0):
    t = inner_sampler()
    model_out = model(t)
    validation_constraint(model_out, real_solution(t))

In [ ]:
adam_phase = qewton.optim.OptimizationPhase(
    optimizer=qewton.optim.Adam(),
    lr=0.001,
    max_iterations=1000,
)

trainer = qewton.optim.GraphBasedTrainer(
    optimization_phases=adam_phase,
    graphs=[boundary_graph, ode_graph, validation_pipeline],
    training_objectives=[boundary_constraint, constraint],
    callbacks=[
        qewton.optim.ModelParameterSizeCallback([model])
    ],
    device="cuda:0",
)


In [ ]:
save_path = "results"   

tuner = qewton.optim.tuner.GridSearchTuner(
    trainer,
    tuning_objectives=[validation_constraint],
    trial_number=40,
    devices=[
        "cuda:0",
        "cuda:1",
        "cuda:2",
        "cuda:3",
    ],
    save_path=save_path,
    trials_per_device=1,
)
tuner.run()

In [ ]:
tuning_analyzer = qewton.optim.TuningAnalyzer(save_path)

In [ ]:
tuning_analyzer.best_run()

In [ ]:
tuning_analyzer.param_importance()

In [ ]:
fig = tuning_analyzer.parallel_coordinates_plot(
    axes=["FCN Hidden Neurons", "FCN Hidden Layers", "Inner Sampler Points", "Parameter size [MB]"],
    objectives=["ValidationConstraint"],
    log_axes=["ValidationConstraint"]
    )
fig.show()